In [1]:
import json
import os
import pandas as pd
import torch
import numpy as np

from collections import defaultdict
from dotenv import load_dotenv
from openai import OpenAI
from threading import Thread
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TextIteratorStreamer
from util import *
load_dotenv()
pd.set_option('display.max_rows', None)


model_ = "gpt-5.1"
log_name_ = "credit_seed52_ratio0.3_polluted_withLabel"
chunk_size_ = 1

llm = llm_call(model_version = model_, api_key= os.getenv("API_KEY"))
df_new, cases_json = build_event_jsons(log_name = f"./dataset/{log_name_}.csv", chunk_cases = chunk_size_)

# 🏷️ Polluted Labels Step 1: Identification of Candidates

This phase focuses on identifying **Polluted Labels** (Mutable Qualifiers)—activity names that contain inconsistent elements such as unique IDs, timestamps, or system codes (e.g., `Order_10293`, `Step-20240101`).

### 🎯 Objective
The goal is to identify and extract clusters consisting of "Polluted Labels" and their corresponding "Clean" boiler-plate versions. This allows for noise removal (ID stripping) in subsequent stages of the data cleaning pipeline.

### ⚖️ Selection Logic: The "Pair" Rule
1. **Identify Polluted Labels**: Detect labels containing variable identifiers based on specific patterns (8+ digit numeric IDs, 6+ alphanumeric codes, or delimiters like `_`, `-`, `:`, `/`, `#`).
2. **Boiler-plate Context**: Keep a "Clean" version (e.g., `Process`) **ONLY IF** a polluted variant (e.g., `Process_V1`) exists in the same list.
3. **Discard Isolated Labels**: Remove clean labels that do not have any polluted counterparts to prevent over-filtering of standard activities.
4. **Discard Noise**: Ignore pure typos (Distorted) or semantic synonyms that do not follow the "Fixed Base + Variable ID" pattern.

### 📤 Expected Output
- **found**: Boolean indicating if any polluted patterns were detected.
- **data**: A refined list of candidate strings (both clean bases and their polluted variants) for the next processing step.

In [2]:


activity_list_json = json.dumps(df_new['activity'].unique().tolist(), indent=4, ensure_ascii=False)

SYSTEM_PROMPT_STEP1 = """
You are an expert Process Mining Data Pre-processor.
Your goal is to filter a raw list of activity names based on specific criteria provided in the User Prompt.

### KNOWLEDGE BASE: IMPERFECTION PATTERNS
Use these definitions to identify which labels belong to which category.

1.  **Polluted Labels (Mutable Qualifiers):**
    Labels that share a immutable boiler-plate text but differ due to mutable text (e.g., embedded IDs or codes).
    * **Detection Criteria:**
        * **Long Numeric IDs:** 8+ digits (e.g., `20260122`, `9988776655`).
        * **Mixed Codes:** 6+ alphanumeric chars (e.g., `XJ9281`, `Ref_A1B2C3`).
        * **Delimiters:** Attached via `_`, `-`, `:`, `/`, `#`, `.`, or space.

2.  **Distorted Labels (Character-Level Corruption):**
    Labels containing specific character-level corruptions (typos, OCR faults) of a canonical form. Unlike synonyms, these are "Noise".
    * **Detection Criteria:**
        1.  **Case Mutation:** Identical spelling, different capitalization (e.g., "Open" vs "open" vs "OPEN").
        2.  **Character Omission:** Exactly ONE missing character (e.g., "Invoce" vs "Invoice").
        3.  **Character Insertion:** Exactly ONE extra character (e.g., "Innvoice" vs "Invoice").
        4.  **Character Transposition:** Two adjacent characters swapped (e.g., "Ivnoice" vs "Invoice").
        5.  **Keyboard Proximity:** Exactly ONE character substituted by a QWERTY neighbor (e.g., "Invoicr" vs "Invoice").

3. **Synonymous Labels (Semantic Equivalence):**
   Labels that are syntactically different (often substantially) but share the same semantic meaning and represent the exact same business process step. 
   * **Detection Criteria (Ontology Rules):**
        1. **Linguistic & Domain Synonyms:** Different words representing the same concept within the process context (e.g., "Ship Item" vs "Dispatch Goods", "DrSeen" vs "Medical Assign").
        2. **Phrase Variation (Verb/Object Shift):** Labels sharing a core component (usually the Object) while using synonymous verbs or adjectives (e.g., "Create Invoice" vs "Generate Invoice", "Start instance" vs "Start process").
        3. **Grammatical Transformation:** Changing parts of speech (Noun ↔ Verb) or sentence structure while retaining the core meaning (e.g., "Give approval" vs "Approve", "Conduct analysis" vs "Analyze").
        4. **Containment & Refinement:** One label is a concise or verbose version of the other, often omitting non-essential adjectives, prepositions, or 'online/offline' qualifiers (e.g., "Receive signed contract" vs "Receive contract", "Register for course" vs "Register course").

### GLOBAL INSTRUCTION
- **Role:** Function as a logic engine. Do not assume all imperfections exist.
- **Priority:** The strict filtering logic in the **User Prompt** overrides general definitions here.
- **OUTPUT FORMAT:** Always return valid JSON as requested by the User Prompt.

"""

USER_PROMPT_POLLUTED_STEP1 = f"""
### TASK: Filter Data for 'Polluted Label' Candidates

**OBJECTIVE:**
Analyze the provided **INPUT DATA(activity list)** and extract labels exhibiting **Polluted Labels (Mutable Qualifiers)**.
You must **identify and collect** the following components for the output data:
1. All **Polluted Labels** found (labels with embedded IDs, dates, or complex codes).
2. Any **Immutable Boiler-plate Text (Clean Label)** found **ONLY IF** they correspond to a Polluted Label present in the list.

**STRICT FILTERING LOGIC:**
1. **Identify Polluted Labels:** Look for labels containing variable identifiers based on the System Prompt's criteria (IDs, Codes) and **KEEP** them regardless of whether their template base exists (e.g., Keep `["Step_X99"]` even if `"Step"` is missing).
2. **KEEP Clean/Immutable Boiler-plate Text Labels :** Check for the "Clean" version of any detected Polluted Label and **KEEP** it **ONLY IF** it corresponds to a Polluted Label present in the list (e.g., Keep `"Step"` only if `"Step_X99"` exists).
3. **DISCARD Isolated Clean Labels:** If a label is "Clean" but has **NO** polluted variants in the provided list, **REMOVE IT**. (e.g., Remove `"Start Process"` if no `"Start Process_123"` exists).
4. **DISCARD Distorted/Synonymous:** Remove labels that are purely 'Distorted Labels' or 'Synonymous Labels' if they are not part of a 'Polluted Labels' pattern.

**EDGE CASE HANDLING:**
- If NO Polluted labels are found (i.e., all labels are clean, distorted, or synonyms):
    - Return strictly `[]` (with "found": false).
- **Finding NOTHING is a valid result.** Do not include isolated clean labels just to populate the list.

**INPUT DATA:**
{activity_list_json}

***OUTPUT FORMAT GUIDELINES (PERFORMANCE OPTIMIZED)***
Return a JSON Object with two keys:
1. "found": Boolean (true if polluted labels exist, false otherwise).
2. "data": List of strings.

**Example (Found):**
{{ "found": true, "data": ["Case_20240101", "Case_20240102", "Case"] }}

**Example (Not Found - SPEED PRIORITY):**
{{ "found": false, "data": [] }}

**CONSTRAINT:**
- Determine the "found" value FIRST. If false, output `[]` for data immediately.
- Output ONLY the JSON.
"""


In [3]:
prompt = [{"role": "system", "content": SYSTEM_PROMPT_STEP1},
          {"role":"user","content": USER_PROMPT_POLLUTED_STEP1}] 

polluted_step1 = llm_gen(model_version = model_, model_instance = llm, prompt = prompt)
polluted_step1_json = json.dumps(polluted_step1['data'], indent=4, ensure_ascii=False)


In [6]:
for k, v in polluted_step1.items():
    print(f"{k}:")
    if isinstance(v, list):  # 값이 리스트인 경우
        if v:  # 리스트가 비어 있지 않을 때
            print(*v, sep="\n")
        else:  # 리스트가 비어 있을 때
            print(" (empty list)")
    else:  # 'found'와 같이 리스트가 아닌 경우
        print(f" {v}")
    print() # 가독성을 위한 한 줄 띄우기

found:
 True

data:
Check for completeness:Clerk-000005
Perform checks:Clerk-000006
Make decision:20230929 093300179000
Notify accept:20230929 093736568000
Deliver card:Manager-000001
Check for completeness:Clerk-000006
notify reject.20230929 141521082000
Check for completeness.Clerk-000006
Make decision.20230929 143451646000
Deliver card>20230929 143747189000
Check for completeness.20230929 134121583000
Request info>20230929 140114504000
info received_20230929 141900830000
notify reject:Manager-000002
review request received_20231002 041717921000
Make decision.20231002 061153392000
Notify accept-20231003 032559935000
Check for completeness>Clerk-000002
time out_20231003 115610111000
Perform checks.20231003 171413441000
review request received:20231003 222544700000
Request info:Manager-000004
Request info.Manager-000002
Check for completeness:20231004 035516795000
Request info:Manager-000002
Deliver card:20231004 014213653000
Check for completeness.20231004 013634384000
Perform checks.

# 🏷️ Polluted Labels Step 2: Contextual Flow Abstraction

This phase focuses on summarizing the structural context of potential **Polluted Labels**. Since polluted labels (e.g., `Order_101`, `Order_102`) often share the same process position, we abstract their neighbor activities to confirm they belong to the same functional stage.

### 🎯 Objective
The goal is to transform raw predecessor and successor activity lists into high-level **Process Stage Names**. This abstraction helps the LLM recognize that different activities are functionally identical if they are triggered by and result in the same business phases.

### ⚖️ Abstraction Logic
1. **Flow Extraction**: Use the Directly Follows Graph (DFG) to identify immediate predecessors and successors.
2. **Significance Filtering**: Only neighbors representing more than **5%** of the total flow are kept to eliminate infrequent noise paths.
3. **Semantic Summarization**: The LLM analyzes the neighbor lists (e.g., `["Scan Barcode", "Print Label"]`) and abstracts them into a single descriptive string (e.g., `"Logistics Preparation"`).

### 📤 Expected Output
- A JSON object where `predecessors` and `successors` for each candidate activity are represented as **Summarized Strings** rather than raw lists.
2. Code Cell (복사해서 코드 셀에 붙여넣으세요)

In [7]:
def get_polluted_context(df: pd.DataFrame,
                        case_col: str = 'case_id',
                        time_col: str = 'timestamp',
                        act_col: str = 'activity',
                        filter_list: set = None):
    df_pm4py = df[[case_col, time_col, act_col]].copy()
    df_pm4py.rename(columns={
        case_col: "case:concept:name",
        time_col: "time:timestamp",
        act_col: "concept:name"
    }, inplace=True)
    df_pm4py["time:timestamp"] = pd.to_datetime(df_pm4py["time:timestamp"], errors="coerce")
    dfg, start_activities, end_activities = pm4py.discover_dfg(df_pm4py)
    def get_activity_context(activity, dfg_dict):
        predecessors = {k[0]: v for k, v in dfg_dict.items() if k[1] == activity}
        successors = {k[1]: v for k, v in dfg_dict.items() if k[0] == activity}
        total_pred = sum(predecessors.values())
        total_succ = sum(successors.values())
        def format_to_list(dist_dict, total):
            if total == 0: return []
            items = [(k, v/total) for k, v in dist_dict.items() if (v/total) >= 0.05]
            items.sort(key=lambda x: x[1], reverse=True)
            return [k for k, v in items]
        return format_to_list(predecessors, total_pred), format_to_list(successors, total_succ)
    all_activities = sorted(df[act_col].unique())
    flow_data_list = []
    for act in all_activities:
        if filter_list is not None and act not in filter_list:
            continue
        pred, succ = get_activity_context(act, dfg)
        flow_data_list.append({
            'activity': act,
            'predecessors': pred, # 이제 리스트입니다 ['A', 'B']
            'successors': succ    # 이제 리스트입니다 ['C', 'D']
        })
        
    json_flow_context = json.dumps(flow_data_list, indent=2, ensure_ascii=False)
    return json_flow_context

    
##print(synonym_clusters_json)

target_activities_set = polluted_step1['data']
polluted_context_json = get_polluted_context(
    df=df_new,          
    filter_list=target_activities_set 
)
print(polluted_context_json)

[
  {
    "activity": "Check for completeness",
    "predecessors": [
      "info received",
      "review request received"
    ],
    "successors": [
      "Request info",
      "Perform checks"
    ]
  },
  {
    "activity": "Check for completeness.20230929 134121583000",
    "predecessors": [],
    "successors": [
      "Request info>20230929 140114504000"
    ]
  },
  {
    "activity": "Check for completeness.20231004 013634384000",
    "predecessors": [],
    "successors": [
      "Perform checks.Clerk-000005"
    ]
  },
  {
    "activity": "Check for completeness.20231010 030110727000",
    "predecessors": [],
    "successors": [
      "Perform checks:20231010 032355179000"
    ]
  },
  {
    "activity": "Check for completeness.20231010 103604942000",
    "predecessors": [
      "info received-nan"
    ],
    "successors": [
      "Perform checks"
    ]
  },
  {
    "activity": "Check for completeness.20231030 010950359000",
    "predecessors": [],
    "successors": [
      "Per

In [ ]:
SYSTEM_PROMPT_POLLUTED_STEP2  = """
You are an expert Process Mining Analyst.
Your goal is to summarize lists of activity labels into a single, descriptive **Process Stage Name**.

### CORE TASK
You will be given an activity and its lists of **Predecessors** (incoming flow) and **Successors** (outgoing flow).
You must analyze the labels in each list and determine the **Common Business Phase** they represent.

### SUMMARIZATION LOGIC (ABSTRACTION)
1. **Identify the Core Action:** Look at the verbs and objects in the list.
2. **Ignore Noise:** Disregard synonyms, typos, and minor variations.
3. **Formulate a Summary:** Create a short, natural language phrase that encapsulates the collective meaning.

### EXAMPLES (Demonstration Only)
- **Input List:** `["Wrap package", "Box items", "Pack goods", "Containerize"]`
- **Output Summary:** "Packaging Phase"

- **Input List:** `["MRI Scan", "X-Ray taken", "Blood test results"]`
- **Output Summary:** "Medical Diagnosis Stage"

- **Input List:** `["Ticket Resolved", "Issue Fixed", "Close Ticket", "Problem Solved"]`
- **Output Summary:** "Ticket Resolution"

### GLOBAL INSTRUCTION
- **Input:** JSON object with `activity`, `predecessors` (list), and `successors` (list).
- **Output:** JSON object where `predecessors` and `successors` are converted to **Strings** (Summaries).
"""

USER_PROMPT_POLLUTED_STEP2 = f"""
### TASK: Summarize Contextual Flow Lists

**OBJECTIVE:**
Analyze the **INPUT DATA**. Replace the list of strings in `predecessors` and `successors` with a **Single Summarized String** describing that process stage.

**STRICT EXECUTION STEPS:**
1. **Iterate** through every activity in the input.
2. **Analyze Predecessors:**
   - Read the list of predecessor labels.
   - Abstract their common meaning into one short phrase (e.g., "Quality Check Phase").
   - **Replace** the list with this string.
3. **Analyze Successors:**
   - Read the list of successor labels.
   - Abstract their common meaning into one short phrase.
   - **Replace** the list with this string.

**INPUT DATA:**
{polluted_context_json}

***OUTPUT FORMAT GUIDELINES***
Return a JSON Object with a single key `"summarized_context"`.
The value must be a list of objects where `predecessors` and `successors` are **STRINGS**, not lists.

**Example Output (Mental Model):**
{{
  "summarized_context": [
    {{
      "activity": "Ship Item",
      "predecessors": "Packaging Phase",     // Was ["Box items", "Wrap package"...]
      "successors": "Delivery Initiation"    // Was ["Truck loaded", "Dispatch"...]
    }},
    {{
      "activity": "Handle Error",
      "predecessors": "System Failure",      // Was ["Crash", "Server Down"...]
      "successors": "Recovery Process"       // Was ["Reboot", "Restart"...]
    }}
  ]
}}

**Constraint:**
- Output **ONLY** the JSON object.
"""

prompt = [{"role": "system", "content": SYSTEM_PROMPT_POLLUTED_STEP2},
          {"role":"user","content": USER_PROMPT_POLLUTED_STEP2}] 

polluted_step2 = llm_gen(model_version = model_, model_instance = llm, prompt = prompt)
polluted_step2_json = json.dumps(polluted_step2["summarized_context"], indent=2, ensure_ascii=False)
print(polluted_step2_json)

# 🏷️ Polluted Labels Step 3: Fuzzy Context & Pattern Mapping

In this final identification stage, we perform **Fuzzy Context Matching** combined with **Textual Root Analysis** to map polluted variants back to their original "Clean" labels.

### 🎯 Objective
The goal is to create a definitive mapping dictionary where each **Clean Label (Key)** is associated with its **Polluted Variants (Value)**. This ensures that activities like `Approve_User01` and `Approve_User02` are correctly identified as the same functional step: `Approve`.

### ⚖️ Detection Logic: The Two-Factor Rule
1. **Condition 1: Context Similarity (Validation)**:
    - The LLM compares summarized predecessors and successors.
    - Even if the wording differs slightly (e.g., "Info Received" vs. "Receipt of Info"), they are treated as the same process point if the **core meaning** matches.
2. **Condition 2: Textual Containment (Root Check)**:
    - The "Clean Label" must be the root phrase or substring of the variant.
    - Within a contextually similar group, the **Shortest/Simplest** string is designated as the Clean Root.

### 📤 Expected Output
- A strict JSON object where:
    - **Key**: The Clean (Canonical) Label.
    - **Value**: A list of identified Polluted Variants (IDs, codes, etc.).

In [ ]:
SYSTEM_PROMPT_POLLUTED_STEP3 = """
You are an expert Process Mining Data Cleaner specializing in **Polluted Label Detection**.
Your goal is to identify the "Clean Label" (Canonical Form) and map all its "Polluted Variants" based on Context and Text Patterns.

### INPUT DATA
You will receive objects with:
1. `activity`: The label.
2. `predecessors`: A summarized string (Input Context).
3. `successors`: A summarized string (Output Context).

### KNOWLEDGE BASE: POLLUTED LABELS (Mutable Qualifiers)
A label is "Polluted" if it consists of a **Clean Root** followed by mutable text (Noise) like IDs or codes.
- **Detection Criteria:**
    - **Pattern:** `[Clean Label] + [Delimiter] + [ID/Code]`
    - **Delimiters:** `_`, `-`, `:`, `/`, `#`, `.`, or space.
    - **Noise Examples:** Long numeric IDs (8+ digits), Mixed Codes (e.g., `XJ9281`), User IDs (e.g., `Clerk-001`).

### DETECTION LOGIC: FUZZY CONTEXT & PATTERN
To map a Polluted Variant to a Clean Label, BOTH conditions must be met:

**CONDITION 1: CONTEXT SIMILARITY (Validation)**
- Compare `predecessors_A` vs `predecessors_B` AND `successors_A` vs `successors_B`.
- **Do not look for exact string matches.**
- **Rule:** The Clean Label and its Polluted Variant must share the **Same Process Context**.
    - *Reasoning:* If "Check_01" and "Check_02" are the same step, they must happen at the same point in the process.

**CONDITION 2: TEXTUAL CONTAINMENT (Root Check)**
- **Rule:** The "Clean Label" must be a substring or the root phrase of the "Polluted Variant".
- **LOGIC:** Within a contextually similar group, the **Shortest / Simplest** string is usually the Clean Label.

### GLOBAL INSTRUCTION
- **Output:** A JSON object where **Key** = Clean Label, **Value** = List of Polluted Variants.
- **Constraint:** Only include pairs where actual pollution is detected. Do not output clean labels that have no variants.
"""

USER_PROMPT_POLLUTED_STEP3 = f"""
### TASK: Clean vs. Polluted Mapping

**OBJECTIVE:**
Analyze the **INPUT DATA**. Identify "Clean Labels" and group their "Polluted Variants" based on the System Prompt's criteria.
**Key Instruction:** Be flexible with context descriptions. Focus on the **Core Meaning**.

**STRICT EXECUTION STEPS:**

1.  **Group by Context:**
    - Look at activities that share **Semantically Similar Predecessors AND Successors**.
    - (Use the "Fuzzy Context" logic: e.g., "Info Received" ≈ "Receipt of Info").

2.  **Identify Clean Root:**
    - Inside each context group, find the label that serves as the **Clean Root**.
    - *Hint:* It is usually the shortest string without numbers or special codes (e.g., "Check" vs "Check_01").

3.  **Map Variants:**
    - Identify other labels in the group that follow the pattern `Clean Root + Delimiter + Code`.
    - Verify they match the **Polluted Definition** (IDs, Mixed Codes).

4.  **Construct Output:**
    - Create a map: `{{ "Clean Label": ["Polluted_Var_1", "Polluted_Var_2"] }}`.

**INPUT DATA (Summarized Context):**
{polluted_step2_json}

***OUTPUT FORMAT GUIDELINES***
Return a strict JSON Object. Keys are strings, Values are lists of strings.

**Example Logic (Mental Model):**
- **Data:**
    * A: "Approve" (Context: X)
    * B: "Approve_Manager1" (Context: X)
    * C: "Approve_Manager2" (Context: X)
- **Analysis:**
    * Context Match: All share Context X.
    * Root Check: "Approve" is the shortest root. B and C contain "Approve" + "_" + ID.
- **Result:** `{{ "Approve": ["Approve_Manager1", "Approve_Manager2"] }}`

**Constraint:**
- Output **ONLY** the JSON object.
"""

prompt = [{"role": "system", "content": SYSTEM_PROMPT_POLLUTED_STEP3},
          {"role":"user","content": USER_PROMPT_POLLUTED_STEP3}] 

polluted_step3 = llm_gen(model_version = model_, model_instance = llm, prompt = prompt)
polluted_step3_json = json.dumps(polluted_step3, indent=4, ensure_ascii=False)


In [ ]:
df_polluted = df_new[df_new['label'].notna()].copy()
df_polluted['clean_activity'] = df_polluted['label'].str.extract(r'\((.*?)\)')

polluted_answer = (
    df_polluted.groupby('clean_activity')['activity']
    .unique()
    .apply(list)
    .to_dict()
)
print("------------------PREDICTION------------------")
print(json.dumps(polluted_step3, indent=4, ensure_ascii=False))
print("------------------ANSWER------------------")
print(json.dumps(polluted_answer, indent=4, ensure_ascii=False))

In [ ]:
def evaluate_polluted_results(answer_dict, predict_dict):
    ans_keys = set(answer_dict.keys())
    pred_keys = set(predict_dict.keys())
    
    tp_keys = ans_keys.intersection(pred_keys)
    fp_keys = pred_keys - ans_keys
    fn_keys = ans_keys - pred_keys
    
    key_precision = len(tp_keys) / len(pred_keys) if pred_keys else 0
    key_recall = len(tp_keys) / len(ans_keys) if ans_keys else 0
    key_f1 = (2 * key_precision * key_recall) / (key_precision + key_recall) if (key_precision + key_recall) else 0
    
    all_v_f1 = []
    all_v_precision = []
    all_v_recall = []

    for key in tp_keys:
        ans_vals = set(answer_dict[key])
        pred_vals = set(predict_dict[key])
        
        tp_v = ans_vals.intersection(pred_vals)
        
        v_prec = len(tp_v) / len(pred_vals) if pred_vals else 0
        v_reca = len(tp_v) / len(ans_vals) if ans_vals else 0
        v_f1 = (2 * v_prec * v_reca) / (v_prec + v_reca) if (v_prec + v_reca) else 0
        
        all_v_precision.append(v_prec)
        all_v_recall.append(v_reca)
        all_v_f1.append(v_f1)
        
    avg_v_precision = sum(all_v_precision) / len(tp_keys) if tp_keys else 0
    avg_v_recall = sum(all_v_recall) / len(tp_keys) if tp_keys else 0
    avg_v_f1 = sum(all_v_f1) / len(tp_keys) if tp_keys else 0

    print("-" * 50)
    print("      [Polluted Pattern Detection Evaluation Report]")
    print("-" * 50)
    print(f"1. Key Selection (Clean Activity Identification)")
    print(f"   - Precision : {key_precision:.4f}")
    print(f"   - Recall    : {key_recall:.4f}")
    print(f"   - F1-Score  : {key_f1:.4f}")
    print("-" * 50)
    print(f"2. Value Selection (Polluted Variation Mapping Accuracy - Average)")
    print(f"   - Avg Precision : {avg_v_precision:.4f}")
    print(f"   - Avg Recall    : {avg_v_recall:.4f}")
    print(f"   - Avg F1-Score  : {avg_v_f1:.4f}")
    print("-" * 50)
    print(f"   * Analyzed Keys: {len(tp_keys)} matched / {len(ans_keys)} total")
    print("-" * 50)

    return {
        "key_metrics": {"precision": key_precision, "recall": key_recall, "f1": key_f1},
        "value_metrics": {"avg_precision": avg_v_precision, "avg_recall": avg_v_recall, "avg_f1": avg_v_f1}
    }
    
print(evaluate_polluted_results(polluted_answer, polluted_step3))